In [ ]:
# ============================================================
# BIASCOPE — Notebook 10 (v5): Advanced demo
# Ensemble uncertainty, classifier-consistent LLM explanation, two-stage rewrite with meaning guard,
# multi-source comparison, cross-lingual check, precise related-coverage search, report export
# Requires: src/biascope_core.py, models/final/xlmr_lora_mbib_babe_s{42,43,44},
#           models/final/uncertainty_config.json (NB11), GPU runtime recommended, internet for coverage search
# ============================================================
!pip -q uninstall -y torchao
!pip -q install -U gradio peft transformers accelerate bitsandbytes feedparser sentence-transformers requests

import datetime, html as _html, importlib, json, re, sys, urllib.parse, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests, feedparser
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")
BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
sys.path.insert(0, str(BASE_DIR / "src"))
import biascope_core as core
importlib.reload(core)

LLM_ID = "Qwen/Qwen3-8B"          # memory/speed problem ho to "Qwen/Qwen3-4B"
SEEDS = (42, 43, 44)
SIM_THR = 0.70                    # coverage: LaBSE similarity for a 'closely matching' headline (not validated)
SIM_MIN_REWRITE = 0.80            # rewrite: minimum LaBSE similarity to the original (meaning guard)
REFINE_THR = 0.50                 # refine the rewrite if it is still scored above this
REPORT_DIR = Path("/content/reports"); REPORT_DIR.mkdir(exist_ok=True)

# ---------------- Ensemble classifier + uncertainty config ----------------
cfg_path = BASE_DIR / "models" / "final" / "uncertainty_config.json"
assert cfg_path.exists(), "uncertainty_config.json nahi mila — pehle NB11 chalao."
CFG = json.load(open(cfg_path))
MARGIN, SSTD = float(CFG["margin_delta"]), float(CFG["seed_std_sigma"])
bms = []
for s in SEEDS:
    m = core.BiasModel(adapter_dir=BASE_DIR / "models" / "final" / f"xlmr_lora_mbib_babe_s{s}")
    m.T = float(CFG["temperatures"][str(s)])
    bms.append(m)
bm0 = bms[0]
print(f"Ensemble ready ({len(bms)} models) | thresholds: margin<{MARGIN:.3f} or seed_std>{SSTD:.3f} | device={core.DEVICE}")

XL = None
xl_path = BASE_DIR / "outputs" / "nb11_uncertainty_consistency" / "cross_lingual_consistency.csv"
if xl_path.exists():
    XL = pd.read_csv(xl_path, index_col=0).iloc[:, 0]

def ens_probs(texts):
    P = np.stack([m.probs(texts) for m in bms], 0)
    return P.mean(0), P.std(0)

def status_of(p, sd):
    if abs(p - 0.5) < MARGIN or sd > SSTD:
        return "uncertain"
    return "biased" if p >= 0.5 else "neutral"

STATUS_STYLE = {
    "biased": ("#c0392b", "BIASED", "The classifier is confident that the text is biased. Check the evidence words below."),
    "neutral": ("#2c6fbb", "NEUTRAL", "No strong bias signal. This does not prove that the text is unbiased, accurate or true."),
    "uncertain": ("#b45309", "UNCERTAIN", "Low confidence: human review is recommended."),
}

def badge_html(p, sd, per_model=None):
    color, title, sub = STATUS_STYLE[status_of(p, sd)]
    pm = (" | per-model P: " + " / ".join(f"{x:.2f}" for x in per_model)) if per_model is not None else ""
    return (f"<div style='border-left:8px solid {color};padding:10px 14px;background:#f7f7f7;border-radius:6px'>"
            f"<div style='font-size:22px;font-weight:700;color:{color}'>{title}</div>"
            f"<div>{sub}</div>"
            f"<div style='font-size:13px;color:#444;margin-top:4px'>Calibrated P(biased) = <b>{p:.2f}</b> "
            f"(ensemble of {len(bms)} models, disagreement σ = {sd:.3f}{pm})</div>"
            f"<div style='font-size:12px;color:#777;margin-top:2px'>Low disagreement between models does not guarantee "
            f"that the prediction is correct.</div></div>")

# ---------------- Local LLM ----------------
class LocalLLM:
    def __init__(self, model_id):
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto").eval()

    @torch.no_grad()
    def generate(self, prompts, max_new_tokens=300):
        texts = [self.tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                              add_generation_prompt=True, enable_thinking=False) for p in prompts]
        enc = self.tok(texts, return_tensors="pt", padding=True).to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                  temperature=None, top_p=None, top_k=None, pad_token_id=self.tok.pad_token_id)
        n_in = enc["input_ids"].shape[1]
        return [self.tok.decode(o[n_in:], skip_special_tokens=True) for o in out]

def parse_field(txt, field):
    if not txt:
        return None
    t = re.sub(r"<think>.*?</think>", "", txt, flags=re.S).strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.M).strip()
    m = re.search(r"\{.*\}", t, flags=re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            v = obj.get(field) if isinstance(obj, dict) else None
            if v:
                return str(v).strip()
        except Exception:
            pass
    m = re.search(r'"%s"\s*:\s*"(.*?)"\s*[,}]' % re.escape(field), t, flags=re.S)
    return m.group(1).strip() if m else None

llm = None
if core.DEVICE == "cuda":
    try:
        print("Loading", LLM_ID, "...")
        llm = LocalLLM(LLM_ID)
        print("LLM ready | GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))
    except Exception as e:
        llm = None
        print("LLM could not be loaded:", repr(e)[:200])
else:
    print("CPU runtime: AI explanation/rewrite/summary disabled (classifier, highlights, comparison, coverage still work).")

LLM_CACHE = {}
def llm_gen(prompts, max_new_tokens=300):
    todo = [p for p in dict.fromkeys(prompts) if (p, max_new_tokens) not in LLM_CACHE]
    if todo:
        for p, o in zip(todo, llm.generate(todo, max_new_tokens=max_new_tokens)):
            LLM_CACHE[(p, max_new_tokens)] = o
    return [LLM_CACHE[(p, max_new_tokens)] for p in prompts]

# ---------------- Prompts ----------------
def explain_prompt_v2(text, p, status, influences, out_lang):
    verdict = {"biased": "BIASED", "neutral": "NEUTRAL", "uncertain": "UNCERTAIN"}[status]
    return ("You interpret the output of a media-bias classifier for a reader. The classifier, not you, decided the verdict, "
            "and you must stay consistent with it.\n"
            "Text: " + json.dumps(text, ensure_ascii=False) + "\n"
            "Classifier verdict: " + verdict + " (calibrated probability of bias = " + format(p, ".2f") + ")\n"
            "Words that statistically influenced the score: " + "; ".join(influences) + "\n"
            "Rules: (1) If the verdict is NEUTRAL, explain why the text reads as factual or neutral and do NOT claim that it "
            "is biased; numbers, prices, percentages and statistics are factual information, not bias by themselves. "
            "(2) If the verdict is UNCERTAIN, say that the signals are mixed and name what is ambiguous. "
            "(3) If the verdict is BIASED, point to specific loaded wording. "
            "(4) The influential words are model influences, not proof of bias; mention them only when they support the verdict. "
            "(5) Do not add facts that are not in the text.\n"
            "Task: write 2-3 sentences in " + core.LANG_NAME[out_lang] + ".\n"
            'Return JSON: {"explanation": "..."}')

def refine_prompt(original, rewrite, remaining, lang):
    return ("You rewrote a text in a neutral tone, but a media-bias classifier still flags the rewrite.\n"
            "Original: " + json.dumps(original, ensure_ascii=False) + "\n"
            "Current rewrite: " + json.dumps(rewrite, ensure_ascii=False) + "\n"
            "Words in the current rewrite that the classifier still treats as loaded: " + ", ".join(remaining) + "\n"
            "Task: write ONE improved rewrite that removes the remaining loaded wording, emotional intensifiers, one-sided "
            "characterisations and opinion stated as fact, while keeping every factual claim, name and number of the original. "
            "Do not add facts. Keep the same language (" + core.LANG_NAME[lang] + ").\n"
            'Return JSON: {"rewrite": "..."}')

def framing_prompt(rows):
    parts = []
    for r in rows:
        parts.append(f"- {r['name']} (mean P(biased)={r['mean']:.2f}, flagged sentences={r['flag']:.0%}). "
                     f"Excerpt: {json.dumps(r['excerpt'], ensure_ascii=False)}")
    return ("Below are news texts, presumably about the same event, with scores from a media-bias classifier.\n"
            + "\n".join(parts) +
            "\nTask: (a) in 1-2 sentences, state the points that all texts share; (b) in 2-3 sentences, describe how they differ "
            "in wording, emphasis and framing (loaded words, emotional tone). Use ONLY information in the excerpts. "
            "Do not judge which text is true and do not add facts. Write in English.\n"
            'Return JSON: {"shared_points": "...", "differences": "..."}')

# ---------------- Related-coverage search (precise matching) ----------------
from sentence_transformers import SentenceTransformer
labse = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")     # CPU: GPU memory nahi khaata
print("LaBSE ready (CPU)")

def labse_sim(a, b):
    e = labse.encode([a[:300], b[:300]], normalize_embeddings=True)
    return float(e[0] @ e[1])

STOP = set("the a an of in on at to for and or but is are was were be been by with from as that this it its his her their they he she "
           "we you i not no have has had will would can could may might said says say after before over under into than then also more "
           "most new है हैं का की के में से को पर ने और यह वह भी तो था थे थी कर किया गया गए हो एक इस उस लिए कि".split())

def content_tokens(text):
    return {t for t in re.findall(r"[\w\u0900-\u097F]+", text.lower()) if len(t) > 2 and t not in STOP}

def make_queries(text, lang):
    words = re.sub(r"\s+", " ", text).strip().split()
    q1 = " ".join(words[:14])
    if lang == "en":
        ents = [w.strip(".,;:!?\"'()“”’") for w in words[1:] if w[:1].isupper() or any(ch.isdigit() for ch in w)]
        q2 = " ".join(dict.fromkeys([e for e in ents if e]))[:120]
    else:
        q2 = " ".join(words[:8])
    return [q for q in dict.fromkeys([q1, q2]) if q.strip()]

def google_news(query, lang, n=15):
    hl, gl, ceid = ("hi", "IN", "IN:hi") if lang == "hi" else ("en-IN", "IN", "IN:en")
    url = f"https://news.google.com/rss/search?q={urllib.parse.quote(query)}&hl={hl}&gl={gl}&ceid={ceid}"
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (BIASCOPE research demo)"}, timeout=12)
    resp.raise_for_status()
    feed = feedparser.parse(resp.content)
    out = []
    for e in feed.entries[:n]:
        title = e.get("title", "")
        src = e.get("source", None)
        src_name = src.get("title", "") if isinstance(src, dict) else ""
        head, outlet = (title.rsplit(" - ", 1) if " - " in title else (title, src_name))
        out.append(dict(outlet=(src_name or outlet).strip(), headline=head.strip(),
                        date=str(e.get("published", ""))[:16], link=e.get("link", "")))
    return out

def cov_box(color, title, body):
    return (f"<div style='border-left:8px solid {color};padding:8px 12px;background:#f7f7f7;border-radius:6px;margin-bottom:6px'>"
            f"<b style='color:{color}'>{title}</b><div style='font-size:14px'>{body}</div></div>")

VERIFY_BOX = cov_box("#555", "CLAIM NOT INDEPENDENTLY VERIFIED",
                     "BIASCOPE retrieves related coverage; it does not fact-check. Absence of coverage is not evidence that a "
                     "claim is false, and presence of coverage is not proof that it is true. Outlets are not rated. "
                     "Key terms of the text are sent to the news-search service.")

def cov_table(items):
    rows = "".join(
        f"<tr><td>{_html.escape(u['outlet'])}</td>"
        f"<td><a href='{_html.escape(u['link'], quote=True)}' target='_blank'>{_html.escape(u['headline'])}</a></td>"
        f"<td>{_html.escape(u['date'])}</td><td>{u['sim']:.2f}</td><td>{u['ov']}</td></tr>" for u in items)
    return ("<table style='width:100%;border-collapse:collapse;font-size:14px;margin-top:6px' border='1' cellpadding='4'>"
            "<tr style='background:#eee'><th>Outlet</th><th>Headline (link)</th><th>Date</th><th>Similarity</th><th>Shared terms</th></tr>"
            f"{rows}</table>")

def find_coverage(text, lang=None, top_n=8):
    text = (text or "").strip()
    if not text:
        return "Please enter a text."
    lang = lang or core.detect_lang(text)
    items, errors = [], []
    for q in make_queries(text, lang):
        try:
            items += google_news(q, lang)
        except Exception as e:
            errors.append(str(e)[:80])
    seen, uniq = set(), []
    for it in items:
        key = (it["outlet"], it["headline"][:60])
        if it["headline"] and key not in seen:
            seen.add(key); uniq.append(it)
    if not uniq:
        why = f"Search failed ({errors[0]})." if errors else "The search returned no articles."
        return cov_box("#666", "NO CLOSE COVERAGE FOUND", why + " This does not mean the claim is false.") + VERIFY_BOX
    claim_tok = content_tokens(text[:300])
    min_ov = 2 if len(claim_tok) >= 6 else 1
    emb_c = labse.encode([text[:300]], normalize_embeddings=True)
    emb_h = labse.encode([u["headline"] for u in uniq], normalize_embeddings=True, batch_size=32)
    for u, sm in zip(uniq, (emb_h @ emb_c.T).ravel()):
        u["sim"] = float(sm)
        u["ov"] = len(content_tokens(u["headline"]) & claim_tok)
        u["close"] = u["sim"] >= SIM_THR and u["ov"] >= min_ov
    uniq.sort(key=lambda x: -x["sim"])
    close = [u for u in uniq if u["close"]]
    loose = [u for u in uniq if not u["close"]][:top_n]
    outlets = sorted({u["outlet"] for u in close if u["outlet"]})
    if len(outlets) >= 2:
        box = cov_box("#2e7d32", "MATCHING COVERAGE FOUND",
                      f"{len(outlets)} outlets carry closely matching headlines (similarity ≥ {SIM_THR:.2f} and ≥ {min_ov} shared key "
                      f"term(s)): {_html.escape(', '.join(outlets[:8]))}.")
    elif len(outlets) == 1:
        box = cov_box("#b45309", "LIMITED MATCHING COVERAGE", f"Only one outlet carries a closely matching headline ({_html.escape(outlets[0])}).")
    else:
        box = cov_box("#666", "NO CLOSE COVERAGE FOUND",
                      "No headline matched closely. This is not evidence that the claim is false "
                      "(it may be very recent, local, an opinion piece, or not indexed).")
    out = box + (cov_table(close[:top_n]) if close else "")
    if loose:
        out += ("<details style='margin-top:6px'><summary style='cursor:pointer'>Show loosely related results (below the match threshold)</summary>"
                + cov_table(loose) + "</details>")
    return out + VERIFY_BOX

# ---------------- Helpers: highlighting, sentences, warnings ----------------
def render_html_en(words, scores):
    mx = max(float(np.max(np.abs(scores))) if len(scores) else 0.0, 1e-9)
    parts = []
    for w, s in zip(words, scores):
        a = min(abs(float(s)) / mx, 1.0)
        color = "255,80,80" if s > 0 else "70,130,255"
        style = "background: rgba(%s,%.2f); padding:1px 3px; border-radius:3px; margin:1px;" % (color, 0.05 + 0.75 * a)
        parts.append('<span title="attribution %+.2f" style="%s">%s</span>' % (float(s) / mx, style, _html.escape(w)))
    legend = ('<div style="font-size:12px;margin-bottom:6px">'
              '<span style="background:rgba(255,80,80,.6);padding:1px 4px">red = pushes towards biased</span> '
              '<span style="background:rgba(70,130,255,.6);padding:1px 4px">blue = pushes towards neutral</span> '
              '(hover a word for its attribution)</div>')
    return legend + '<div style="font-size:17px;line-height:2.1">' + " ".join(parts) + "</div>"

def top_words_html(ex, k=8):
    sc = np.asarray(ex["scores"], float)
    words = ex["words"]
    if len(sc) == 0:
        return ""
    mx = max(float(np.max(np.abs(sc))), 1e-9)
    idx = np.argsort(-np.abs(sc))[:k]
    rows = "".join(f"<tr><td>{_html.escape(words[i])}</td><td>{sc[i] / mx:+.2f}</td>"
                   f"<td>{'towards biased' if sc[i] > 0 else 'towards neutral'}</td></tr>" for i in idx)
    return ("<table border='1' cellpadding='4' style='border-collapse:collapse;font-size:14px'>"
            "<tr style='background:#eee'><th>Word</th><th>Attribution (normalised)</th><th>Direction</th></tr>" + rows + "</table>"
            "<p style='font-size:12px;color:#666'>Attribution shows how strongly a word influenced the model's score; "
            "it is not proof of bias.</p>")

HINGLISH_WORDS = set("hai hain nahi nahin kya mein ka ki ke ko ne par se tha thi aur yeh woh toh bhi kiya gaya raha rahe liye lekin "
                     "abhi kuch koi jab apne uska unka".split())
def is_romanized_hinglish(text):
    if core.detect_lang(text) != "en":
        return False
    toks = re.findall(r"[a-z]+", text.lower())
    if len(toks) < 4:
        return False
    hits = sum(t in HINGLISH_WORDS for t in toks)
    return hits >= 2 and hits / len(toks) >= 0.15

def warn_box(msg):
    return f"<div style='border-left:6px solid #b45309;padding:6px 10px;background:#fff7ed;margin-bottom:6px'>{msg}</div>"

def split_sentences(text):
    parts = re.split(r"(?<=[.!?।])\s+", text.strip())
    return [p for p in parts if len(p.split()) >= 3]

def sentence_table(text):
    sents = split_sentences(text) or [text.strip()]
    p, sd = ens_probs(sents)
    df = pd.DataFrame({"sentence": sents, "P(biased)": np.round(p, 3), "σ": np.round(sd, 3),
                       "status": [status_of(a, b) for a, b in zip(p, sd)]})
    df["rank"] = df["P(biased)"].rank(ascending=False, method="min").astype(int)
    return df, p, sd

def bar_figure(labels, p, status):
    colors = {"biased": "#e85d5d", "neutral": "#6b9bd1", "uncertain": "#e3a23c"}
    fig, ax = plt.subplots(figsize=(6, max(2, 0.45 * len(labels))))
    ax.barh(range(len(labels)), p, color=[colors[s] for s in status])
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.axvline(0.5, color="gray", ls="--", lw=0.8)
    ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_xlabel("P(biased)  (red=biased, blue=neutral, amber=uncertain)")
    plt.tight_layout()
    return fig

def rewrite_table_html(cands, chosen):
    rows = "".join(f"<tr><td>{_html.escape(c['name'])}{' ✓ (shown)' if c['name'] == chosen else ''}</td>"
                   f"<td>{c['p']:.2f}</td><td>{c['sim']:.2f}</td><td>{_html.escape(c['text'][:160])}</td></tr>" for c in cands)
    return ("<table border='1' cellpadding='4' style='border-collapse:collapse;font-size:14px;width:100%'>"
            "<tr style='background:#eee'><th>Version</th><th>P(biased)</th><th>Meaning similarity</th><th>Text</th></tr>" + rows + "</table>")

def build_report(text, p, sd, pm, warn, hl_html, words_tab, cov_html, expl, rewrite, delta, rw_table, df):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    path = REPORT_DIR / f"biascope_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
    llm_note = "Open-source LLM (Qwen3-8B, local)" if llm is not None else "not used"
    body = f"""<html><head><meta charset='utf-8'><title>BIASCOPE report</title></head>
<body style='font-family:sans-serif;max-width:900px;margin:30px auto;line-height:1.5'>
<h1>BIASCOPE — Bias Investigation Report</h1>
<p style='color:#555'>Generated {ts} | Classifier: XLM-R + LoRA (MBIB + BABE), 3-model ensemble, temperature-calibrated | LLM: {llm_note}</p>
<h2>Input text</h2><p style='background:#f4f4f4;padding:10px'>{_html.escape(text)}</p>
<h2>1. Classifier verdict and evidence</h2>{badge_html(p, sd, pm)}{warn}
{hl_html}{words_tab}
<h2>2. LLM interpretation (may over-read; the classifier verdict takes precedence)</h2><p>{_html.escape(expl)}</p>
<h2>3. Neutral rewrite</h2><p>{_html.escape(rewrite) if rewrite else '(not generated)'}</p><p>{_html.escape(re.sub(r'[*_]', '', delta))}</p>{rw_table}
<h2>4. Related news coverage</h2>{cov_html}
<h2>5. Sentence-wise scores</h2>{df.to_html(index=False, escape=True)}
<h2>Limitations</h2><p>The score is a statistical bias signal, not a verdict of truth, and a NEUTRAL verdict does not mean that a text is
unbiased or accurate. Hindi training labels were inherited from English data through machine translation, and the Hindi evaluation set
was annotated by an LLM. Rewrite neutrality is measured by the classifier (refinement uses its feedback, so the drop is optimistic)
and is not human-verified. Related-coverage results are retrieval aids, not fact-checks. Texts longer than 160 tokens are truncated for
the highlight view.</p>
</body></html>"""
    path.write_text(body, encoding="utf-8")
    return str(path)

# ---------------- Tab 1: Analyze ----------------
def analyze(text, use_llm, use_web, expl_choice):
    text = (text or "").strip()
    empty = pd.DataFrame(columns=["sentence", "P(biased)", "σ", "status", "rank"])
    if not text:
        return "", "", "", "", "", "Please enter some text.", "", "", "", empty, None, None
    lang = core.detect_lang(text)
    Pm = np.stack([m.probs([text]) for m in bms], 0)[:, 0]
    p, s = float(Pm.mean()), float(Pm.std())
    status = status_of(p, s)
    ex = bm0.explain(text, steps=24)
    n_tok = len(bm0.tok(text)["input_ids"])
    warn = ""
    if is_romanized_hinglish(text):
        warn += warn_box("Romanised Hinglish detected: the model is much less reliable on romanised text "
                         "(measured AUC ≈ 0.56–0.58). Treat the verdict with caution.")
    if n_tok > core.MAX_LEN:
        warn += warn_box(f"The text is longer than {core.MAX_LEN} tokens; the highlights cover only the beginning. "
                         "See the sentence-wise scores.")
    hl = render_html_en(ex["words"], ex["scores"])
    words_tab = top_words_html(ex)
    df, ps, ss = sentence_table(text)
    fig = bar_figure([f"S{i + 1}" for i in range(len(df))], ps, df["status"].tolist()) if len(df) >= 2 else None

    expl, rewrite, delta, rw_table = "", "", "", ""
    if use_llm and llm is not None:
        try:
            out_lang = "en" if str(expl_choice).startswith("English") else lang
            sc = np.asarray(ex["scores"], float)
            idx = np.argsort(-np.abs(sc))[:6]
            infl = [f"{ex['words'][i]} ({'towards biased' if sc[i] > 0 else 'towards neutral'})" for i in idx]
            outs = llm_gen([explain_prompt_v2(text, p, status, infl, out_lang), core.rewrite_prompt(text, lang)])
            expl = parse_field(outs[0], "explanation") or "(an explanation could not be generated)"
            rw1 = parse_field(outs[1], "rewrite")
            if rw1:
                cands = []
                def add(name, rw):
                    pr, _ = ens_probs([rw])
                    cands.append(dict(name=name, text=rw, p=float(pr[0]), sim=labse_sim(text, rw)))
                add("v1", rw1)
                if status != "neutral" and cands[0]["p"] >= REFINE_THR:
                    exr = bm0.explain(rw1, steps=16)
                    scr = np.asarray(exr["scores"], float)
                    rem = [exr["words"][i] for i in np.argsort(-scr)[:5] if scr[i] > 0]
                    if rem:
                        rw2 = parse_field(llm_gen([refine_prompt(text, rw1, rem, lang)])[0], "rewrite")
                        if rw2 and rw2.strip() != rw1.strip():
                            add("v2 (refined)", rw2)
                ok = [c for c in cands if c["sim"] >= SIM_MIN_REWRITE]
                final = min(ok, key=lambda c: c["p"]) if ok else max(cands, key=lambda c: c["sim"])
                rewrite = final["text"]
                flag = "" if ok else "  ⚠ meaning may have changed (similarity below threshold)"
                delta = (f"**P(biased): {p:.2f} → {final['p']:.2f}  ({final['p'] - p:+.2f}) | meaning similarity {final['sim']:.2f}**{flag}  \n"
                         f"_(measured by the classifier; refinement uses classifier feedback, so the drop is optimistic; "
                         f"neutrality is not human-verified)_")
                rw_table = rewrite_table_html(cands, final["name"])
        except Exception as e:
            expl = f"AI explanation failed: {str(e)[:160]}"
            torch.cuda.empty_cache()
    elif use_llm:
        expl = "AI explanation is unavailable in this session (a GPU runtime is required)."
    else:
        expl = "_AI interpretation is off (tick the checkbox above to enable it)._"

    if use_web:
        try:
            cov = find_coverage(text, lang)
        except Exception as e:
            cov = cov_box("#666", "COVERAGE CHECK FAILED", _html.escape(str(e)[:160])) + VERIFY_BOX
    else:
        cov = "<i>Related-coverage search is off (tick the checkbox to enable it).</i>" + VERIFY_BOX
    expl_plain = re.sub(r"^_|_$", "", expl)
    report = build_report(text, p, s, Pm, warn, hl, words_tab, cov, expl_plain, rewrite, delta, rw_table, df)
    return badge_html(p, s, Pm), warn, hl, words_tab, cov, expl, rewrite, rw_table, delta, df, fig, report

# ---------------- Tab 2: Multi-source comparison ----------------
def compare(n1, t1, n2, t2, n3, t3, n4, t4, use_llm):
    arts = [((n or f"Source {i}").strip(), (t or "").strip())
            for i, (n, t) in enumerate(zip([n1, n2, n3, n4], [t1, t2, t3, t4]), 1) if (t or "").strip()]
    cols = ["Source", "#sent", "Mean P(biased)", "Max P(biased)", "% flagged", "% uncertain", "Top words", "Most biased sentence"]
    if len(arts) < 2:
        return pd.DataFrame(columns=cols), None, "Please enter at least two articles."
    rows, summ_rows = [], []
    for name, t in arts:
        df, ps, ss = sentence_table(t)
        i = int(np.argmax(ps))
        ex = bm0.explain(df["sentence"].iloc[i], steps=16)
        sc = np.asarray(ex["scores"], float)
        top = [ex["words"][j] for j in np.argsort(-sc)[:3]]
        stt = df["status"]
        rows.append({"Source": name, "#sent": len(df), "Mean P(biased)": round(float(ps.mean()), 3),
                     "Max P(biased)": round(float(ps.max()), 3), "% flagged": round(float((ps >= 0.5).mean() * 100), 1),
                     "% uncertain": round(float((stt == "uncertain").mean() * 100), 1),
                     "Top words": ", ".join(top), "Most biased sentence": df["sentence"].iloc[i][:160]})
        summ_rows.append(dict(name=name, mean=float(ps.mean()), flag=float((ps >= 0.5).mean()), excerpt=t[:500]))
    out = pd.DataFrame(rows, columns=cols)
    fig, ax = plt.subplots(figsize=(6, max(2.2, 0.6 * len(out))))
    ax.barh(range(len(out)), out["Mean P(biased)"], color="#6b9bd1", label="mean")
    ax.scatter(out["Max P(biased)"], range(len(out)), color="#c0392b", zorder=3, label="max sentence")
    ax.set_yticks(range(len(out))); ax.set_yticklabels(out["Source"]); ax.invert_yaxis()
    ax.axvline(0.5, color="gray", ls="--", lw=0.8); ax.set_xlim(0, 1); ax.set_xlabel("P(biased)"); ax.legend()
    plt.tight_layout()
    if use_llm and llm is not None:
        try:
            txt = llm_gen([framing_prompt(summ_rows)], max_new_tokens=400)[0]
            sh, df_ = parse_field(txt, "shared_points"), parse_field(txt, "differences")
            summ = ((f"**Shared points:** {sh}\n\n" if sh else "") + (f"**Differences in wording and framing:** {df_}" if df_ else "")
                    ) or "(a summary could not be generated)"
            summ += "\n\n_LLM-generated from short excerpts; not validated. Do not treat it as a fact-check._"
        except Exception as e:
            summ = f"Framing summary failed: {str(e)[:160]}"
    else:
        summ = "_LLM summary is off (tick the checkbox to enable it)._"
    summ += "\n\n_Article score = mean of sentence-level scores; the classifier was trained on single sentences._"
    return out, fig, summ

# ---------------- Tab 3: Cross-lingual check ----------------
_mt = {}
def translate_en_hi(text):
    if not _mt:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        name = "facebook/nllb-200-distilled-600M"
        tok = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn")
        dt = torch.float16 if core.DEVICE == "cuda" else torch.float32
        model = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=dt).to(core.DEVICE).eval()
        _mt.update(tok=tok, model=model, tgt=tok.convert_tokens_to_ids("hin_Deva"))
    sents = split_sentences(text) or [text]
    enc = _mt["tok"](sents, return_tensors="pt", padding=True, truncation=True, max_length=200).to(core.DEVICE)
    with torch.no_grad():
        gen = _mt["model"].generate(**enc, forced_bos_token_id=_mt["tgt"], num_beams=4, max_new_tokens=200,
                                    no_repeat_ngram_size=4, repetition_penalty=1.1)
    return " ".join(_mt["tok"].batch_decode(gen, skip_special_tokens=True))

def xling(en_text, hi_text, auto):
    en_text, hi_text = (en_text or "").strip(), (hi_text or "").strip()
    if not en_text:
        return "Please enter the English text.", hi_text
    if auto or not hi_text:
        try:
            hi_text = translate_en_hi(en_text)
        except Exception as e:
            return f"Translation failed: {str(e)[:160]}", hi_text
    pe, se = ens_probs([en_text]); ph, sh = ens_probs([hi_text])
    pe, se, ph, sh = float(pe[0]), float(se[0]), float(ph[0]), float(sh[0])
    d = ph - pe
    same = (pe >= 0.5) == (ph >= 0.5)
    verdict = ("Consistent: both versions get the same decision and similar scores." if same and abs(d) < 0.25 else
               "Shift detected: the score or decision changes across languages. Loaded words are often lost or "
               "introduced in translation, so compare the two word lists before trusting either score.")
    study = ""
    if XL is not None:
        study = (f"<p style='font-size:13px;color:#555'>On 999 held-out English/Hindi sentence pairs (BABE test), the decision "
                 f"agreed for {float(XL['label_agreement']):.1%} of pairs and the score changed by more than 0.25 for "
                 f"{float(XL['share_abs_delta_gt_0p25']):.1%} of pairs.</p>")
    exE, exH = bm0.explain(en_text, steps=16), bm0.explain(hi_text, steps=16)
    res = (f"<h4>English</h4>{badge_html(pe, se)}{top_words_html(exE, 6)}"
           f"<h4>Hindi</h4>{badge_html(ph, sh)}{top_words_html(exH, 6)}"
           f"<p><b>Δ = P(hi) − P(en) = {d:+.2f}</b></p><p>{verdict}</p>{study}")
    return res, hi_text

# ---------------- Tab 4: Source check ----------------
def source_check(text):
    return find_coverage(text, None, top_n=10)

# ---------------- Examples + smoke tests ----------------
EN_CHOICE = "English (recommended)"
EXAMPLES = [
    ["The incompetent, radical-left administration has shamelessly destroyed the economy.", False, False, EN_CHOICE],
    ["The government raised fuel prices by 5% on Tuesday, officials said.", False, False, EN_CHOICE],
    ["सनसनीखेज खुलासा: इस मशहूर अभिनेत्री ने किया चौंकाने वाला खुलासा, सच जानकर आप भी हो जाएंगे हैरान", False, False, EN_CHOICE],
    ["केंद्र सरकार ने सोमवार को आयातित इस्पात पर नया शुल्क लगाने की घोषणा की।", False, False, EN_CHOICE],
]
for t, _, _, _ in EXAMPLES:
    analyze(t, False, False, EN_CHOICE)
    pe, sd = ens_probs([t])
    print(f"Smoke test | {status_of(float(pe[0]), float(sd[0])):9s} | P={float(pe[0]):.2f} σ={float(sd[0]):.3f} | {t[:55]}")
print("Hinglish detector:", is_romanized_hinglish("sansanikhej khulasa: is abhinetri ne kiya chaunkane wala khulasa"),
      "(expected True) |", is_romanized_hinglish("The government raised fuel prices by 5% on Tuesday."), "(expected False)")
cmp_out = compare("A", EXAMPLES[0][0], "B", EXAMPLES[1][0], "", "", "", "", False)
print("Compare smoke test rows:", len(cmp_out[0]))
print("Cross-lingual smoke test OK:", "Δ" in xling(EXAMPLES[1][0], EXAMPLES[3][0], False)[0])
try:
    cov_test = find_coverage("Reserve Bank of India keeps repo rate unchanged", "en")
    print("Coverage smoke test:", "OK (status box rendered)" if "COVERAGE" in cov_test else "check output")
except Exception as e:
    print("Coverage smoke test failed (internet?):", repr(e)[:150])
if llm is not None:
    o = analyze(EXAMPLES[1][0], True, False, EN_CHOICE)
    print("LLM smoke test (neutral text) | explanation:", o[5][:160].replace("\n", " "))
    o = analyze(EXAMPLES[0][0], True, False, EN_CHOICE)
    print("LLM smoke test (biased text)  | delta:", o[8][:100].replace("\n", " "))
plt.close("all")

# ---------------- UI ----------------
llm_label = ("AI interpretation + neutral rewrite (open-source LLM, runs locally)" if llm is not None
             else "AI interpretation + neutral rewrite (unavailable in this session: GPU runtime needed)")
with gr.Blocks(title="BIASCOPE") as demo:
    gr.Markdown("# BIASCOPE — multilingual news bias analysis (English / Hindi)\n"
                "Classifier: XLM-R + LoRA trained on MBIB + BABE (English + machine-translated Hindi); 3-model ensemble, "
                "temperature-calibrated, with uncertainty-aware verdicts. Highlights = Integrated Gradients. "
                "The interpretation, rewrite and framing summary come from an open-source LLM (Qwen3-8B) running locally; "
                "**the bias verdict always comes from the classifier**. Related coverage is retrieved from a news search and "
                "is **not a fact-check**.")
    with gr.Tab("Analyze"):
        inp = gr.Textbox(lines=5, label="Text (English or Hindi)", placeholder="Paste a headline, a sentence or a short article…")
        use_llm = gr.Checkbox(value=False, label=llm_label, interactive=llm is not None)
        expl_lang = gr.Radio(["English (recommended)", "Same language as input (Hindi explanations are lower quality)"],
                             value=EN_CHOICE, label="Interpretation language")
        use_web = gr.Checkbox(value=False, label="Find related news coverage (web search; works best with a headline-like first sentence)")
        btn = gr.Button("Analyze", variant="primary")
        gr.Markdown("## Classifier verdict and evidence")
        badge_out = gr.HTML()
        warn_out = gr.HTML()
        html_out = gr.HTML()
        words_out = gr.HTML()
        gr.Markdown("## LLM interpretation (may over-read; the classifier verdict takes precedence)")
        expl_out = gr.Markdown()
        gr.Markdown("## Neutral rewrite")
        rewrite_out = gr.Textbox(label="Neutral rewrite (final)", lines=4)
        delta_out = gr.Markdown()
        rwtab_out = gr.HTML()
        gr.Markdown("## Related news coverage (corroboration check, not a fact-check)")
        cov_out = gr.HTML()
        sent_df = gr.Dataframe(label="Sentence-wise scores (rank 1 = highest P(biased))", interactive=False)
        sent_plot = gr.Plot(label="Bias timeline (sentence-wise)")
        report_out = gr.File(label="Download investigation report (HTML; use the browser's Print → Save as PDF)")
        btn.click(analyze, [inp, use_llm, use_web, expl_lang],
                  [badge_out, warn_out, html_out, words_out, cov_out, expl_out, rewrite_out, rwtab_out, delta_out,
                   sent_df, sent_plot, report_out])
        gr.Examples(examples=EXAMPLES, inputs=[inp, use_llm, use_web, expl_lang])
    with gr.Tab("Compare sources"):
        gr.Markdown("Paste 2–4 texts about the same event (optionally name the sources).")
        boxes = []
        for i in range(1, 5):
            with gr.Row():
                n = gr.Textbox(label=f"Source {i} name", scale=1)
                t = gr.Textbox(label=f"Text {i}", lines=3, scale=4)
            boxes += [n, t]
        cmp_llm = gr.Checkbox(value=False, label="LLM summary of shared points and framing differences (local)", interactive=llm is not None)
        cbtn = gr.Button("Compare", variant="primary")
        cmp_df = gr.Dataframe(interactive=False)
        cmp_plot = gr.Plot()
        cmp_summary = gr.Markdown()
        cbtn.click(compare, boxes + [cmp_llm], [cmp_df, cmp_plot, cmp_summary])
    with gr.Tab("Cross-lingual check"):
        gr.Markdown("Does the bias signal survive translation? Enter an English text; the Hindi version can be pasted or auto-translated (NLLB-200).")
        x_en = gr.Textbox(lines=3, label="English text")
        x_hi = gr.Textbox(lines=3, label="Hindi text (optional)")
        x_auto = gr.Checkbox(value=True, label="Auto-translate English → Hindi")
        xbtn = gr.Button("Check consistency", variant="primary")
        x_out = gr.HTML()
        xbtn.click(xling, [x_en, x_hi, x_auto], [x_out, x_hi])
    with gr.Tab("Source check"):
        gr.Markdown("Find recent news coverage of a claim or headline (English or Hindi). Results show which outlets carry a closely "
                    "matching headline, with links. **This does not verify truth**: absence of coverage is not evidence that a claim "
                    "is false.")
        sc_in = gr.Textbox(lines=3, label="Headline or claim")
        sc_btn = gr.Button("Search coverage", variant="primary")
        sc_out = gr.HTML()
        sc_btn.click(source_check, [sc_in], [sc_out])
    gr.Markdown("**Limitations:** the bias score is a *statistical bias signal*, not a verdict of truth, and a NEUTRAL verdict does not "
                "mean a text is unbiased or accurate. Hindi training labels were inherited from MBIB/BABE through machine translation, "
                "and the Hindi evaluation set was annotated by an LLM. The model is weak on romanised Hinglish (AUC ≈ 0.56–0.58). "
                "The classifier was trained on single sentences; article scores are averages of sentence scores. 'Uncertain' marks the "
                "least confident predictions (thresholds fitted on a validation set); it flags difficult samples, not domain shift as such. "
                "Rewrite neutrality is measured by the classifier and is not human-verified (refinement uses its feedback, so the drop is "
                "optimistic). Related-coverage results come from a news search (key terms of the input are sent to that service); they "
                "are retrieval aids, the match threshold is not validated, and outlets are not rated.")

demo.queue().launch(share=True, debug=True)